In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType
import pandas as pd
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.functions import hour, dayofweek
from pyspark.ml.feature import StringIndexer
import time

start_time = time.time()
# 初始化 Spark 会话
spark = SparkSession.builder \
    .appName("wucaishendata") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
    .config("spark.driver.memory", "200g") \
    .config("spark.executor.memory", "32g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

# Optional: configure additional S3 settings
spark._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "s3.amazonaws.com")  # or a specific region endpoint
spark._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "true")
spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")  # for some S3-compatible services

# Path to the large CSV file (can be local or on S3)
csv_input_path = "s3a://your-bucket-name/input/large_file.csv"

# Read the large CSV file
df = spark.read.csv(csv_input_path, header=True, inferSchema=True)

# Perform transformations (optional)
# For example: df = df.select("column1", "column2")

# Write the DataFrame to S3 in Parquet (or CSV) format
output_path = "s3a://your-bucket-name/output/processed_data/"

df.write.mode("overwrite").parquet(output_path)  # or use .csv(output_path) for CSV

# Stop the Spark session
spark.stop()


# 读取数据
df = spark.read.csv("wucaishen202503.csv", header=True, inferSchema=True)
df = df.filter((col("flag") != -8.0) & (col("productid") != "B26"))